# Step 1: Build evaluation tasks from your repository

Public coding benchmarks are not your code. In this notebook you turn a repository's own merged pull
requests into evaluation tasks; in Step 2 you run coding agents on them.

This notebook teaches the steps; the work is done by the scripts in `step1-build-tasks/`, which you can
run the same way outside a notebook. As a sample it uses a small public repository,
[pallets/itsdangerous](https://github.com/pallets/itsdangerous), and one of its pull requests.

**What you will do**
1. Describe how the repository's tests run (a profile).
2. Turn a merged pull request into a coding task.
3. Look at what the task holds, and what the agent can and cannot see.
4. Calibrate the task on AgentCore: find which hidden tests the change makes pass, and check the task
   with two reference agents.

## Two kinds of tasks

- **PR review**: the agent reviews a change and is graded against the review the change really got.
  These are built with eval-harvest's review workflow (see `step1-build-tasks/README.md`).
- **Coding**: the agent makes a merged pull request's change, starting from the commit before it, and
  the tests that pull request added or changed grade it. This notebook builds this kind.

Why a merged pull request makes a good task: it holds a real change *and* the tests a developer
wrote to check it. Hide the change and the new tests, give the agent the title, and the tests become
the answer key.

## 0. Before you start

Run this notebook from the `Coding Assistant` folder. You need `git`, [uv](https://docs.astral.sh/uv/),
and for the calibration step the AgentCore setup of Step 2 (its `.venv` with Harbor, and
`harbor_env/setup.py bootstrap`) in `us-east-1`.

In [ ]:
!git --version && uv --version
!aws sts get-caller-identity --query Arn --output text

## 1. Describe how the tests run

Every repository runs its tests differently, so the task needs a **profile**: the base image, how to
install the dependencies (at build time: the agent and the grader have no internet later), and the
test command. The grader fills in `{files}`, the hidden test files, and `{junit}`, where the runner
writes its JUnit XML report: the grader only ever reads that report, which is why any test runner that
writes JUnit works.

`agent_setup` installs the agents' own tools in the image, for the same reason.

In [ ]:
!cat step1-build-tasks/coding-profile.toml

## 2. Turn a pull request into a task

`build_coding_tasks.sh` clones the repository and runs `eval-harvest code` for each pull request. It
reads only git history: the pull request's merge commit, the commit before it, and what changed.

The change is split in two:
- the **test side** (changed test files): hidden from the agent, restored over whatever the agent
  does to them, and used to grade it;
- the **source side**: what the agent has to write. It is kept as the reference solution, which only
  the `oracle` agent ever applies.

A pull request that changes no tests, or only tests, is skipped with a reason.

In [ ]:
!bash step1-build-tasks/build_coding_tasks.sh https://github.com/pallets/itsdangerous.git 378

## 3. Look at the task

The instruction is the pull request's title, the hidden test files and how they run. It never quotes
the pull request's description, which usually explains how the fix works.

`environment/` is what the agent gets: the repository at the commit before the change, as a single
fresh commit, so there is no git history to read the change from. `tests/` and `solution/` are not in
the agent's image.

In [ ]:
T = "step1-build-tasks/work/dataset/tasks/pallets__itsdangerous__pr378-code"
!cat {T}/instruction.md
!ls -R {T} | head -30

## 4. Calibrate on AgentCore

The pull request touched a test file, but which of its tests does the change make pass? Some passed
before, some may be flaky. That is found by running them, on the same AgentCore path Step 2 uses, with
two reference agents from Harbor:

- `nop` changes nothing: every hidden test as it is before the change;
- `oracle` applies the real change.

A test that fails with `nop` and passes with `oracle` is **fail-to-pass**: it is what the agent has to
make pass. One that passes with both is **pass-to-pass**: the agent must not break it.
`eval-harvest calibrate` writes them into the task, and lists the fail-to-pass tests in the
instruction.

Then the **gate**: `nop` must now score 0 (a task it passes asks for nothing) and `oracle` 1 (a task
it fails cannot be solved).

Grading is in **stages**: the tests run as the agent user and write only a report; a separate,
trusted stage computes the score from that report, as a different user, after every agent process
has ended. So an agent cannot write its own score.

This needs Step 2's setup. If you have not done it yet:

In [ ]:
# Step 2's setup, once (see README.md): Harbor in a venv, then the AWS side for the models you will test.
!uv venv --python 3.12 .venv && uv pip install --python .venv/bin/python 'harbor==0.26.0' 'boto3>=1.44.7'
!PYTHONPATH=$PWD AWS_REGION=us-east-1 .venv/bin/python harbor_env/setup.py bootstrap --agent-model us.anthropic.claude-sonnet-5-5

In [ ]:
!bash step1-build-tasks/calibrate_tasks.sh {T}

## 5. Use your own repository

1. Copy `step1-build-tasks/coding-profile.toml` and change the image, install and test command.
2. Choose merged pull requests that change code and its tests together.
3. Run the two scripts with `PROFILE=<your profile>` and your repository's URL.

Each task that passes the gate is a Harbor task like the samples in `datasets/`: run agents on it in
Step 2 with `harbor run -p step1-build-tasks/work/dataset/tasks ...`.

When you are done with AgentCore, clean up as Step 2 describes (`harbor_env/setup.py teardown`).